# 04 — Sensor robustness and reliability

This notebook asks a different question from the sensor-budget analysis: **what happens when sensor readings become unreliable?**

The comparison includes the three Phase 4 validation-frontier configurations:

- Light;
- Humidity + Light; and
- Temperature + Light + CO2.

They are comparison cases, not deployment finalists. This phase does not make a sensor recommendation.

The notebook reads artifacts from the packaged robustness command and does not retrain models:

```powershell
$env:PYTHONPATH = "$PWD\src"
python -m sensorbudget.robustness.evaluate
```

## 1. Experiment design

The fitted sensor-budget models are evaluated on perturbed copies of Test 1 and Test 2. Every result is compared with the same model's clean baseline.

The scenarios cover:

- occupied darkness and unoccupied lighting;
- random missing feature cells with training-median fallback;
- bounded Gaussian measurement noise;
- complete loss of one sensor with training-median fallback;
- sensors stuck at low or high training-period values; and
- gradual positive and negative calibration drift.

Random missingness and noise are repeated five times with reproducible random seeds. The Light-policy scenarios use the known label to create diagnostic counterfactuals; they are not transformations available during live prediction.

In [ ]:
# Import table-processing, display, and interactive Plotly tools.
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots

PLOTLY_TEMPLATE = "plotly_white"
CANDIDATE_LABELS = {
    "light": "Light",
    "humidity__light": "Humidity + Light",
    "temperature__light__co2": "Temperature + Light + CO2",
}
CANDIDATE_COLORS = {
    "light": "#4C78A8",
    "humidity__light": "#F2A900",
    "temperature__light__co2": "#2A9D8F",
}
SPLIT_DASHES = {"test_1": "solid", "test_2": "dash"}


In [ ]:
# Locate generated artifacts from either the project root or notebooks/.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "models" / "robustness").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

METRICS_PATH = PROJECT_ROOT / "models" / "robustness" / "robustness_metrics.csv"
if not METRICS_PATH.exists():
    raise FileNotFoundError(
        "Missing robustness results. Run "
        "`python -m sensorbudget.robustness.evaluate` first."
    )

# Load scenario-level metrics and create stable display labels.
metrics = pd.read_csv(METRICS_PATH)
metrics["candidate_label"] = metrics["feature_set"].map(CANDIDATE_LABELS)
metrics["split_label"] = metrics["split"].str.replace("_", " ").str.title()
metrics["severity_numeric"] = pd.to_numeric(
    metrics["severity"], errors="coerce"
)

print(f"Loaded {len(metrics)} robustness evaluations.")
display(metrics["scenario_group"].value_counts().rename("rows").to_frame())


## 2. Clean held-out baselines

The clean scores establish the reference point for every degradation value shown later. All three configurations begin with high F1 in both supplied test periods.

In [ ]:
# Plot the unmodified held-out F1 for each comparison configuration.
baseline = metrics.loc[metrics["scenario_group"] == "baseline"].copy()
fig = go.Figure()
for split, split_rows in baseline.groupby("split"):
    fig.add_trace(
        go.Bar(
            x=split_rows["candidate_label"],
            y=split_rows["f1"],
            name=split.replace("_", " ").title(),
            customdata=split_rows[["precision", "recall"]],
            hovertemplate=(
                "%{x}<br>F1: %{y:.3f}"
                "<br>Precision: %{customdata[0]:.3f}"
                "<br>Recall: %{customdata[1]:.3f}<extra></extra>"
            ),
        )
    )
fig.update_layout(
    template=PLOTLY_TEMPLATE,
    title="Clean held-out performance before simulated faults",
    xaxis_title="Physical sensors",
    yaxis_title="F1",
    barmode="group",
    height=500,
)
fig.update_yaxes(range=[0, 1.01], dtick=0.2, showgrid=True)
fig.show()


**Conclusion.** All three configurations achieve an F1 score above 0.95 on both clean held-out datasets, and the differences between them are small. This establishes a strong reference point, but clean performance alone does not tell us how the models behave when sensor inputs become unreliable.

## 3. Lighting-policy failures

These diagnostic interventions deliberately break the relationship between Light and occupancy:

- **Unoccupied but lit:** unoccupied rows receive the median Light value from occupied training rows.
- **Occupied but dark:** occupied rows receive the median Light value from unoccupied training rows.

The interventions are intentionally severe. Their purpose is to reveal shortcut reliance, not estimate the frequency of these situations.

In [ ]:
# Combine clean and Light-policy results so the failure size is visible directly.
policy = metrics.loc[metrics["scenario_group"] == "light_policy"].copy()
policy["scenario_label"] = policy["scenario"].map(
    {
        "unoccupied_lit": "Unoccupied but lit",
        "occupied_dark": "Occupied but dark",
    }
)
clean_for_policy = baseline.copy()
clean_for_policy["scenario_label"] = "Clean baseline"
policy_plot = pd.concat([clean_for_policy, policy], ignore_index=True)
scenario_order = ["Clean baseline", "Unoccupied but lit", "Occupied but dark"]

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Test 1", "Test 2"),
    shared_yaxes=True,
)
for column, split in enumerate(["test_1", "test_2"], start=1):
    split_rows = policy_plot.loc[policy_plot["split"] == split]
    for feature_set, candidate_rows in split_rows.groupby("feature_set"):
        ordered = candidate_rows.set_index("scenario_label").reindex(
            scenario_order
        ).reset_index()
        fig.add_trace(
            go.Bar(
                x=ordered["scenario_label"],
                y=ordered["f1"],
                name=CANDIDATE_LABELS[feature_set],
                marker_color=CANDIDATE_COLORS[feature_set],
                showlegend=column == 1,
                hovertemplate=(
                    CANDIDATE_LABELS[feature_set]
                    + "<br>%{x}<br>F1: %{y:.3f}<extra></extra>"
                ),
            ),
            row=1,
            col=column,
        )
fig.update_layout(
    template=PLOTLY_TEMPLATE,
    title="Performance after breaking the Light–occupancy relationship",
    barmode="group",
    height=560,
    legend_title_text="Physical sensors",
)
fig.update_yaxes(title_text="F1", range=[0, 1.01], dtick=0.2, row=1, col=1)
fig.show()


**Conclusion.** All three configurations deteriorate sharply when the historical relationship between Light and occupancy is reversed. In particular, simulated occupied-but-dark observations reduce F1 to approximately zero, showing that the fitted models depend strongly on the lighting pattern found in the original data.

## 4. Randomly missing readings

A specified fraction of feature cells is removed independently and replaced with medians calculated from the training period. Each severity is repeated five times. The line is the mean F1 and the error bars show one standard deviation across repetitions.

In [ ]:
# Aggregate stochastic missingness repetitions before plotting degradation curves.
missing = metrics.loc[metrics["scenario_group"] == "random_missing"]
missing_summary = (
    missing.groupby(
        ["feature_set", "candidate_label", "split", "severity_numeric"],
        as_index=False,
    )["f1"]
    .agg(["mean", "std"])
    .reset_index()
)

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Test 1", "Test 2"),
    shared_yaxes=True,
)
for column, split in enumerate(["test_1", "test_2"], start=1):
    for feature_set, rows in missing_summary.loc[
        missing_summary["split"] == split
    ].groupby("feature_set"):
        fig.add_trace(
            go.Scatter(
                x=rows["severity_numeric"],
                y=rows["mean"],
                error_y={"type": "data", "array": rows["std"], "visible": True},
                mode="lines+markers",
                name=CANDIDATE_LABELS[feature_set],
                line={"color": CANDIDATE_COLORS[feature_set]},
                showlegend=column == 1,
                hovertemplate=(
                    CANDIDATE_LABELS[feature_set]
                    + "<br>Missing cells: %{x:.0%}<br>Mean F1: %{y:.3f}"
                    "<extra></extra>"
                ),
            ),
            row=1,
            col=column,
        )
fig.update_layout(
    template=PLOTLY_TEMPLATE,
    title="Random missingness with training-median fallback",
    height=520,
    legend_title_text="Physical sensors",
)
fig.update_xaxes(title_text="Missing feature cells", tickformat=".0%")
fig.update_yaxes(title_text="Mean F1", range=[0, 1.01], row=1, col=1)
fig.show()


**Conclusion.** Performance declines as a larger share of readings is replaced by training medians. At 40% missingness, F1 is roughly 0.70 to 0.74 across the configurations and held-out datasets, so median imputation keeps the models operational but does not preserve their clean-data performance.

## 5. Measurement noise

Gaussian noise is scaled by each feature's training-period standard deviation and clipped to its observed training range. A severity of `1.0` therefore means noise with one training standard deviation. Each point averages five deterministic repetitions.

In [ ]:
# Aggregate noise repetitions and compare both held-out periods side by side.
noise = metrics.loc[metrics["scenario_group"] == "gaussian_noise"]
noise_summary = (
    noise.groupby(
        ["feature_set", "candidate_label", "split", "severity_numeric"],
        as_index=False,
    )["f1"]
    .agg(["mean", "std"])
    .reset_index()
)

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Test 1", "Test 2"),
    shared_yaxes=True,
)
for column, split in enumerate(["test_1", "test_2"], start=1):
    for feature_set, rows in noise_summary.loc[
        noise_summary["split"] == split
    ].groupby("feature_set"):
        fig.add_trace(
            go.Scatter(
                x=rows["severity_numeric"],
                y=rows["mean"],
                error_y={"type": "data", "array": rows["std"], "visible": True},
                mode="lines+markers",
                name=CANDIDATE_LABELS[feature_set],
                line={"color": CANDIDATE_COLORS[feature_set]},
                showlegend=column == 1,
                hovertemplate=(
                    CANDIDATE_LABELS[feature_set]
                    + "<br>Noise scale: %{x:.2f} std"
                    "<br>Mean F1: %{y:.3f}<extra></extra>"
                ),
            ),
            row=1,
            col=column,
        )
fig.update_layout(
    template=PLOTLY_TEMPLATE,
    title="Sensitivity to bounded Gaussian sensor noise",
    height=520,
    legend_title_text="Physical sensors",
)
fig.update_xaxes(title_text="Noise standard deviation / training std")
fig.update_yaxes(title_text="Mean F1", range=[0, 1.01], row=1, col=1)
fig.show()


**Conclusion.** Small amounts of noise have limited impact, while stronger noise produces a clear loss in F1. The size of that loss differs between Test 1 and Test 2, and no configuration is consistently the least affected in every condition.

## 6. Complete sensor loss

Complete loss replaces one sensor with its training median for the entire test period. Negative values below show the change from that configuration's own clean F1. This tests a basic fallback policy, not native fault tolerance.

In [ ]:
# Give every configuration–sensor failure its own row.
loss = metrics.loc[metrics["scenario_group"] == "complete_loss"].copy()
loss["failure_label"] = loss["candidate_label"] + " — lose " + loss["sensor"]
failure_order = (
    loss.groupby("failure_label")["f1_delta"].mean().sort_values().index.tolist()
)

fig = go.Figure()
for split, rows in loss.groupby("split"):
    ordered = rows.set_index("failure_label").reindex(failure_order).reset_index()
    # Pre-format the signed change so the hover always shows exactly three decimals.
    ordered["f1_delta_display"] = ordered["f1_delta"].map(lambda value: f"{value:+.3f}")
    fig.add_trace(
        go.Bar(
            x=ordered["f1_delta"],
            y=ordered["failure_label"],
            orientation="h",
            name=split.replace("_", " ").title(),
            customdata=ordered[["f1", "baseline_f1", "f1_delta_display"]],
            hovertemplate=(
                "%{y}<br>F1 change: %{customdata[2]}"
                "<br>Fault F1: %{customdata[0]:.3f}"
                "<br>Clean F1: %{customdata[1]:.3f}<extra></extra>"
            ),
        )
    )
fig.update_layout(
    template=PLOTLY_TEMPLATE,
    title="Change in F1 after complete loss of one sensor",
    xaxis_title="F1 change from clean baseline",
    yaxis_title="Configuration and unavailable sensor",
    barmode="group",
    height=560,
)
fig.add_vline(x=0, line_color="#555555", line_dash="dot")
fig.show()


**Conclusion.** Complete loss of Light reduces F1 to approximately zero for every examined configuration, even when other sensors are available. Losing Humidity, Temperature, or CO2 is generally less damaging, which identifies Light as a shared single point of failure in these fitted models.

## 7. Stuck sensors

Each sensor is frozen at its training-period 5th percentile and 95th percentile. The chart keeps the worse of those two outcomes for every sensor and test period, while the hover identifies whether the low or high value caused it.

In [ ]:
# Select the lower-F1 stuck state for each configuration, split, and sensor.
stuck = metrics.loc[metrics["scenario_group"] == "stuck_sensor"].copy()
worst_stuck = stuck.loc[
    stuck.groupby(["feature_set", "split", "sensor"])["f1"].idxmin()
].copy()
worst_stuck["failure_label"] = (
    worst_stuck["candidate_label"] + " — " + worst_stuck["sensor"]
)
stuck_order = (
    worst_stuck.groupby("failure_label")["f1_delta"]
    .mean()
    .sort_values()
    .index.tolist()
)

fig = go.Figure()
for split, rows in worst_stuck.groupby("split"):
    ordered = rows.set_index("failure_label").reindex(stuck_order).reset_index()
    # Pre-format the signed change so the hover always shows exactly three decimals.
    ordered["f1_delta_display"] = ordered["f1_delta"].map(lambda value: f"{value:+.3f}")
    fig.add_trace(
        go.Bar(
            x=ordered["f1_delta"],
            y=ordered["failure_label"],
            orientation="h",
            name=split.replace("_", " ").title(),
            customdata=ordered[["scenario", "f1", "f1_delta_display"]],
            hovertemplate=(
                "%{y}<br>Worst state: %{customdata[0]}"
                "<br>F1 change: %{customdata[2]}"
                "<br>Fault F1: %{customdata[1]:.3f}<extra></extra>"
            ),
        )
    )
fig.update_layout(
    template=PLOTLY_TEMPLATE,
    title="Worst low-or-high stuck-sensor result",
    xaxis_title="F1 change from clean baseline",
    yaxis_title="Configuration and stuck sensor",
    barmode="group",
    height=560,
)
fig.add_vline(x=0, line_color="#555555", line_dash="dot")
fig.show()


**Conclusion.** A Light sensor stuck at a plausible low or high value can be as damaging as complete sensor loss. This fault is especially important because the input is still numerical and plausible, so ordinary missing-value checks would not detect it.

## 8. Gradual Light drift

Light is gradually offset from zero at the start of the period to the specified fraction of its training standard deviation at the end. Negative values model downward drift; positive values model upward drift. The other sensors remain unchanged.

In [ ]:
# Focus on Light drift because all three configurations share that sensor.
light_drift = metrics.loc[
    (metrics["scenario_group"] == "gradual_drift")
    & (metrics["sensor"] == "Light")
].copy()

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Test 1", "Test 2"),
    shared_yaxes=True,
)
for column, split in enumerate(["test_1", "test_2"], start=1):
    for feature_set, rows in light_drift.loc[
        light_drift["split"] == split
    ].groupby("feature_set"):
        rows = rows.sort_values("severity_numeric")
        fig.add_trace(
            go.Scatter(
                x=rows["severity_numeric"],
                y=rows["f1"],
                mode="lines+markers",
                name=CANDIDATE_LABELS[feature_set],
                line={"color": CANDIDATE_COLORS[feature_set]},
                showlegend=column == 1,
                hovertemplate=(
                    CANDIDATE_LABELS[feature_set]
                    + "<br>Final drift: %{x:+.1f} std"
                    "<br>F1: %{y:.3f}<extra></extra>"
                ),
            ),
            row=1,
            col=column,
        )
fig.update_layout(
    template=PLOTLY_TEMPLATE,
    title="Sensitivity to gradual Light calibration drift",
    height=520,
    legend_title_text="Physical sensors",
)
fig.update_xaxes(title_text="Final Light offset / training std")
fig.update_yaxes(title_text="F1", range=[0, 1.01], row=1, col=1)
fig.show()


**Conclusion.** Light drift causes asymmetric and dataset-dependent degradation: upward and downward shifts do not have identical effects, and Test 2 is generally more sensitive. The multi-sensor models sometimes lose less F1 than the Light-only model, but this does not remove their severe failure under complete or stuck-Light faults.

## 9. Findings and limitations — no sensor recommendation

- **Clean performance hides a shared Light dependency.** All three configurations begin near F1 0.96–0.98, but occupied darkness reduces F1 to approximately zero and unoccupied lighting reduces it to roughly 0.35–0.53.
- **Additional sensors do not automatically provide fallback behavior.** Complete Light loss with median fallback also reduces F1 to approximately zero for every configuration. The current fitted models still rely on Light even when other inputs are available.
- **Scattered missingness degrades more gradually.** At 40% missing feature cells, mean F1 is approximately 0.71–0.74 after training-median fallback.
- **Severe noise is period-dependent.** At one training standard deviation of noise, mean F1 ranges from approximately 0.68 to 0.83; the configuration with more sensors is not uniformly more robust.
- **Low stuck Light and Light drift are material risks.** A low stuck value reproduces the occupied-dark failure, while gradual drift produces smaller but still important degradation.
- **These are simulations, not field failure rates.** Label-targeted lighting interventions are deliberately severe diagnostics, and both test periods come from one room.
- **No sensor configuration is recommended.** The subsequent completed mitigation experiments use training-period validation to test an explicit no-Light fallback, causal routing, missingness indicators, and fault-augmented training.

In [ ]:
# Produce a compact numerical audit of the most important statements above.
audit_rows = []
for scenario_group, scenario, label in [
    ("light_policy", "unoccupied_lit", "Unoccupied but lit"),
    ("light_policy", "occupied_dark", "Occupied but dark"),
    ("complete_loss", "median_fallback", "Complete sensor loss"),
]:
    subset = metrics.loc[
        (metrics["scenario_group"] == scenario_group)
        & (metrics["scenario"] == scenario)
    ]
    if scenario_group == "complete_loss":
        subset = subset.loc[subset["sensor"] == "Light"]
    audit_rows.append(
        {
            "scenario": label,
            "minimum_f1": subset["f1"].min(),
            "maximum_f1": subset["f1"].max(),
        }
    )
audit = pd.DataFrame(audit_rows)
display(audit.round(3))
